# Extract Features & Train Transformer (Colab)

This notebook:
1. Extracts MediaPipe keypoints (42-dim) from video frames
2. Extracts CNN features (512-dim) using your trained model
3. Trains a Transformer on the combined features (554-dim)

## Prerequisites
- Upload `words.zip` to Google Drive (same as CNN training)
- Upload `best_cnn_words.pth` to Google Drive (from CNN training)
- Run with GPU runtime for faster processing

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install MediaPipe
!pip install -q mediapipe

## 1. Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === UPDATE THESE PATHS ===
WORDS_ZIP_PATH = "/content/drive/MyDrive/words.zip"
CNN_MODEL_PATH = "/content/drive/MyDrive/LearningASL_models/best_cnn_words.pth"
# If you saved the model elsewhere, update the path above

import os

# Check files exist
for path, name in [(WORDS_ZIP_PATH, "words.zip"), (CNN_MODEL_PATH, "CNN model")]:
    if os.path.exists(path):
        print(f"✓ Found {name}: {path}")
    else:
        print(f"✗ NOT FOUND {name}: {path}")
        print(f"  Please update the path above!")

In [ ]:
# Extract words.zip
!unzip -q "{WORDS_ZIP_PATH}" -d /content/
!ls /content/words/

In [ ]:
# Count videos
import os
import glob

WORDS_ROOT = "/content/words"

for split in ["train", "val", "test"]:
    split_dir = os.path.join(WORDS_ROOT, split)
    if os.path.isdir(split_dir):
        video_count = 0
        frame_count = 0
        for word in os.listdir(split_dir):
            word_dir = os.path.join(split_dir, word)
            if os.path.isdir(word_dir):
                for video_id in os.listdir(word_dir):
                    video_dir = os.path.join(word_dir, video_id)
                    if os.path.isdir(video_dir):
                        video_count += 1
                        frame_count += len(glob.glob(os.path.join(video_dir, "frame_*.jpg")))
        print(f"{split}: {video_count} videos, {frame_count} frames")

## 2. Define Models

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet50_Weights
import math

# ============ CNN Model ============
class ASL_CNN(nn.Module):
    def __init__(self, num_classes=45):
        super(ASL_CNN, self).__init__()
        self.batch_norm = nn.BatchNorm2d(3)
        self._base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        
        for param in self._base_model.parameters():
            param.requires_grad = False
        for param in self._base_model.layer4.parameters():
            param.requires_grad = True
        
        num_features = self._base_model.fc.in_features
        self._base_model.fc = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.batch_norm(x)
        return self._base_model(x)


class ASL_CNN_FeatureExtractor(nn.Module):
    def __init__(self, trained_asl_cnn):
        super(ASL_CNN_FeatureExtractor, self).__init__()
        self.batch_norm = trained_asl_cnn.batch_norm
        self.resnet_layers = nn.Sequential(
            trained_asl_cnn._base_model.conv1,
            trained_asl_cnn._base_model.bn1,
            trained_asl_cnn._base_model.relu,
            trained_asl_cnn._base_model.maxpool,
            trained_asl_cnn._base_model.layer1,
            trained_asl_cnn._base_model.layer2,
            trained_asl_cnn._base_model.layer3,
            trained_asl_cnn._base_model.layer4,
            trained_asl_cnn._base_model.avgpool
        )
        self.fc_layers = nn.Sequential(
            trained_asl_cnn._base_model.fc[0],
            trained_asl_cnn._base_model.fc[1],
            trained_asl_cnn._base_model.fc[2]
        )

    def forward(self, x):
        x = self.batch_norm(x)
        x = self.resnet_layers(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x  # (batch, 512)


# ============ Transformer Model ============
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class PoseTransformer(nn.Module):
    def __init__(self, input_dim=554, num_classes=45, d_model=256, nhead=4,
                 num_layers=2, dim_feedforward=512, dropout=0.1, max_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.classifier(x)


print("Models defined!")

In [ ]:
# Load the trained CNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load checkpoint
checkpoint = torch.load(CNN_MODEL_PATH, map_location=device)

# Determine number of classes
if '_base_model.fc.3.weight' in checkpoint:
    num_classes = checkpoint['_base_model.fc.3.weight'].shape[0]
else:
    num_classes = 45

print(f"Loading CNN with {num_classes} classes...")

# Create and load model
cnn_model = ASL_CNN(num_classes=num_classes)
cnn_model.load_state_dict(checkpoint)

# Create feature extractor
feature_extractor = ASL_CNN_FeatureExtractor(cnn_model)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Test
test_input = torch.randn(1, 3, 128, 128).to(device)
with torch.no_grad():
    test_output = feature_extractor(test_input)
print(f"Feature extractor test: input {test_input.shape} -> output {test_output.shape}")
print("CNN loaded successfully!")

## 3. Extract Keypoints (MediaPipe)

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

mp_hands = mp.solutions.hands

KEYPOINTS_ROOT = "/content/keypoints_data"


def extract_hand_keypoints(image_bgr, hands):
    """Extract 42-dim keypoints (21 landmarks * 2 coords) from image."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)
    
    if not results.multi_hand_landmarks:
        return None
    
    hand_landmarks = results.multi_hand_landmarks[0]
    coords = []
    for lm in hand_landmarks.landmark:
        coords.extend([lm.x, lm.y])  # Normalized [0, 1]
    
    return np.array(coords, dtype=np.float32)


def extract_keypoints_for_video(video_dir, hands):
    """Extract keypoints for all frames in a video directory."""
    frame_paths = sorted(glob.glob(os.path.join(video_dir, "frame_*.jpg")))
    
    if not frame_paths:
        return None
    
    sequence = []
    for frame_path in frame_paths:
        image_bgr = cv2.imread(frame_path)
        if image_bgr is None:
            keypoints = np.zeros(42, dtype=np.float32)
        else:
            kps = extract_hand_keypoints(image_bgr, hands)
            keypoints = kps if kps is not None else np.zeros(42, dtype=np.float32)
        sequence.append(keypoints)
    
    return np.stack(sequence)  # (T, 42)


print("Keypoint extraction functions defined!")

In [ ]:
# Extract keypoints for all videos
print("Extracting keypoints...")
print("This may take 10-20 minutes...\n")

stats = {"train": 0, "val": 0, "test": 0, "errors": 0}

with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(WORDS_ROOT, split)
        if not os.path.isdir(split_dir):
            continue
        
        words = sorted(os.listdir(split_dir))
        
        for word in tqdm(words, desc=f"{split} keypoints"):
            word_dir = os.path.join(split_dir, word)
            if not os.path.isdir(word_dir):
                continue
            
            for video_id in os.listdir(word_dir):
                video_dir = os.path.join(word_dir, video_id)
                if not os.path.isdir(video_dir):
                    continue
                
                # Output path
                output_dir = os.path.join(KEYPOINTS_ROOT, split, word)
                os.makedirs(output_dir, exist_ok=True)
                output_path = os.path.join(output_dir, f"{video_id}.npy")
                
                # Skip if exists
                if os.path.exists(output_path):
                    stats[split] += 1
                    continue
                
                try:
                    keypoints = extract_keypoints_for_video(video_dir, hands)
                    if keypoints is not None:
                        np.save(output_path, keypoints)
                        stats[split] += 1
                except Exception as e:
                    stats["errors"] += 1

print(f"\nKeypoints extracted:")
print(f"  Train: {stats['train']}")
print(f"  Val: {stats['val']}")
print(f"  Test: {stats['test']}")
print(f"  Errors: {stats['errors']}")

## 4. Extract CNN Features

In [ ]:
from torchvision import transforms
from PIL import Image

CNN_FEATURES_ROOT = "/content/cnn_features"

# Transform for CNN
cnn_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def extract_cnn_features_for_video(video_dir, feature_extractor, device, batch_size=32):
    """Extract CNN features for all frames in a video directory."""
    frame_paths = sorted(glob.glob(os.path.join(video_dir, "frame_*.jpg")))
    
    if not frame_paths:
        return None
    
    # Load and transform frames
    frames = []
    for frame_path in frame_paths:
        try:
            img = Image.open(frame_path).convert('RGB')
            img_tensor = cnn_transform(img)
            frames.append(img_tensor)
        except:
            frames.append(torch.zeros(3, 128, 128))
    
    frames_tensor = torch.stack(frames)  # (T, 3, 128, 128)
    
    # Extract features in batches
    all_features = []
    with torch.no_grad():
        for i in range(0, len(frames_tensor), batch_size):
            batch = frames_tensor[i:i+batch_size].to(device)
            features = feature_extractor(batch)
            all_features.append(features.cpu().numpy())
    
    return np.vstack(all_features)  # (T, 512)


print("CNN feature extraction functions defined!")

In [ ]:
# Extract CNN features for all videos
print("Extracting CNN features...")
print("This may take 15-30 minutes with GPU...\n")

stats = {"train": 0, "val": 0, "test": 0, "errors": 0}

for split in ["train", "val", "test"]:
    split_dir = os.path.join(WORDS_ROOT, split)
    if not os.path.isdir(split_dir):
        continue
    
    words = sorted(os.listdir(split_dir))
    
    for word in tqdm(words, desc=f"{split} CNN features"):
        word_dir = os.path.join(split_dir, word)
        if not os.path.isdir(word_dir):
            continue
        
        for video_id in os.listdir(word_dir):
            video_dir = os.path.join(word_dir, video_id)
            if not os.path.isdir(video_dir):
                continue
            
            # Output path
            output_dir = os.path.join(CNN_FEATURES_ROOT, split, word)
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"{video_id}.npy")
            
            # Skip if exists
            if os.path.exists(output_path):
                stats[split] += 1
                continue
            
            try:
                features = extract_cnn_features_for_video(video_dir, feature_extractor, device)
                if features is not None:
                    np.save(output_path, features)
                    stats[split] += 1
            except Exception as e:
                stats["errors"] += 1

print(f"\nCNN features extracted:")
print(f"  Train: {stats['train']}")
print(f"  Val: {stats['val']}")
print(f"  Test: {stats['test']}")
print(f"  Errors: {stats['errors']}")

## 5. Create Combined Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 60


class CombinedFeatureDataset(Dataset):
    """Dataset that combines keypoints (42-dim) and CNN features (512-dim)."""
    
    def __init__(self, keypoints_root, cnn_features_root, split, max_len=60, classes=None):
        self.max_len = max_len
        self.samples = []  # (kp_path, cnn_path, label_str)
        
        split_dir = os.path.join(keypoints_root, split)
        if not os.path.isdir(split_dir):
            raise FileNotFoundError(f"Split directory not found: {split_dir}")
        
        # Get classes
        found_classes = sorted([d for d in os.listdir(split_dir) 
                                if os.path.isdir(os.path.join(split_dir, d))])
        
        if classes is not None:
            self.classes = classes
        else:
            self.classes = found_classes
        
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        # Collect paired samples
        for cls_name in self.classes:
            kp_cls_dir = os.path.join(keypoints_root, split, cls_name)
            cnn_cls_dir = os.path.join(cnn_features_root, split, cls_name)
            
            if not os.path.isdir(kp_cls_dir):
                continue
            
            kp_files = glob.glob(os.path.join(kp_cls_dir, "*.npy"))
            for kp_path in kp_files:
                filename = os.path.basename(kp_path)
                cnn_path = os.path.join(cnn_cls_dir, filename)
                
                if os.path.isfile(cnn_path):
                    self.samples.append((kp_path, cnn_path, cls_name))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        kp_path, cnn_path, label_str = self.samples[idx]
        label_idx = self.class_to_idx[label_str]
        
        # Load features
        kp_seq = np.load(kp_path).astype(np.float32)      # (T, 42)
        cnn_seq = np.load(cnn_path).astype(np.float32)    # (T, 512)
        
        # Synchronize lengths
        min_len = min(kp_seq.shape[0], cnn_seq.shape[0])
        kp_seq = kp_seq[:min_len]
        cnn_seq = cnn_seq[:min_len]
        
        # Concatenate
        combined = np.concatenate([kp_seq, cnn_seq], axis=1)  # (T, 554)
        
        # Pad/truncate
        T, D = combined.shape
        if T >= self.max_len:
            combined = combined[:self.max_len]
        else:
            padding = np.zeros((self.max_len - T, D), dtype=np.float32)
            combined = np.vstack([combined, padding])
        
        return torch.from_numpy(combined), torch.tensor(label_idx, dtype=torch.long)


print("Dataset class defined!")

In [ ]:
# Create datasets
print("Creating datasets...")

train_dataset = CombinedFeatureDataset(
    keypoints_root=KEYPOINTS_ROOT,
    cnn_features_root=CNN_FEATURES_ROOT,
    split="train",
    max_len=MAX_SEQ_LEN
)

classes = train_dataset.classes
num_classes = len(classes)

val_dataset = CombinedFeatureDataset(
    keypoints_root=KEYPOINTS_ROOT,
    cnn_features_root=CNN_FEATURES_ROOT,
    split="val",
    max_len=MAX_SEQ_LEN,
    classes=classes
)

test_dataset = CombinedFeatureDataset(
    keypoints_root=KEYPOINTS_ROOT,
    cnn_features_root=CNN_FEATURES_ROOT,
    split="test",
    max_len=MAX_SEQ_LEN,
    classes=classes
)

print(f"\nDatasets created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")
print(f"  Classes ({num_classes}): {classes}")

In [ ]:
# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Test a batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nSample batch shape: {sample_batch.shape}")
print(f"Expected: (batch_size, {MAX_SEQ_LEN}, 554)")

## 6. Train Transformer

In [ ]:
# Training functions
import time

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = outputs.max(1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += preds.eq(labels).sum().item()
        total += inputs.size(0)
        
        pbar.set_postfix({"loss": f"{running_loss/total:.4f}", "acc": f"{running_corrects/total:.4f}"})
    
    return running_loss / total, running_corrects / total


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            _, preds = outputs.max(1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += preds.eq(labels).sum().item()
            total += inputs.size(0)
    
    return running_loss / total, running_corrects / total


print("Training functions defined!")

In [ ]:
# Initialize Transformer
INPUT_DIM = 42 + 512  # keypoints + CNN features = 554

transformer = PoseTransformer(
    input_dim=INPUT_DIM,
    num_classes=num_classes,
    d_model=256,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.1,
    max_len=MAX_SEQ_LEN
)
transformer = transformer.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

print(f"Transformer initialized:")
print(f"  Input dim: {INPUT_DIM}")
print(f"  Classes: {num_classes}")
print(f"  Parameters: {sum(p.numel() for p in transformer.parameters()):,}")

In [ ]:
# Training loop
NUM_EPOCHS = 50
TARGET_ACC = 0.95

print("=" * 60)
print(f"Training Transformer for up to {NUM_EPOCHS} epochs")
print(f"Early stopping at {TARGET_ACC:.0%} validation accuracy")
print("=" * 60)

best_acc = 0.0
best_state = None
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    train_loss, train_acc = train_one_epoch(transformer, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(transformer, val_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = transformer.state_dict().copy()
        print(f"-> New best! ({best_acc:.4f})")
    
    if val_acc >= TARGET_ACC:
        print(f"\n*** Target accuracy reached! ***")
        break

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training complete in {total_time/60:.1f} minutes")
print(f"Best Val Accuracy: {best_acc:.4f}")
print("=" * 60)

In [ ]:
# Evaluate on test set
transformer.load_state_dict(best_state)
test_loss, test_acc = validate(transformer, test_loader, criterion, device)

print("=" * 60)
print("TEST RESULTS")
print("=" * 60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 60)

## 7. Save & Download Model

In [ ]:
# Save transformer model
TRANSFORMER_PATH = "/content/best_combined_model.pth"
torch.save(best_state, TRANSFORMER_PATH)
print(f"Transformer saved to: {TRANSFORMER_PATH}")

# Save class mapping
CLASSES_PATH = "/content/transformer_classes.txt"
with open(CLASSES_PATH, "w") as f:
    for i, cls in enumerate(classes):
        f.write(f"{i},{cls}\n")
print(f"Classes saved to: {CLASSES_PATH}")

In [ ]:
# Download models
from google.colab import files

print("Downloading transformer model...")
files.download(TRANSFORMER_PATH)

print("Downloading class mapping...")
files.download(CLASSES_PATH)

In [ ]:
# Also save to Google Drive
DRIVE_SAVE_DIR = "/content/drive/MyDrive/LearningASL_models/"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

!cp {TRANSFORMER_PATH} {DRIVE_SAVE_DIR}
!cp {CLASSES_PATH} {DRIVE_SAVE_DIR}

print(f"\nModels saved to Google Drive: {DRIVE_SAVE_DIR}")
!ls -lh {DRIVE_SAVE_DIR}

---
## Done!

Download these files and put them in your local repo:

```
LearningASL/models/
├── best_cnn_words.pth          # CNN model (from notebook 05)
├── best_combined_model.pth     # Transformer model
└── transformer_classes.txt     # Class mapping
```

The transformer uses combined features:
- Keypoints: 42-dim (MediaPipe hand landmarks)
- CNN: 512-dim (from trained ASL_CNN)
- Total: 554-dim per frame